# FUTURE_ML_01 — Sales & Demand Forecasting

This project forecasts future daily sales from historical business data using time-based feature engineering and Random Forest regression.

**Note:** The included dataset is synthetic and created for this demonstration, so the project is fully reproducible without external data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv('sales_data.csv', parse_dates=['Date']).sort_values('Date')
df.head()

## 1. Data cleaning and exploration

In [ ]:
print(df.info())
print('Missing values:', df.isna().sum().sum())

plt.figure(figsize=(12,4))
plt.plot(df['Date'], df['Sales'])
plt.title('Historical Daily Sales')
plt.xlabel('Date'); plt.ylabel('Sales')
plt.tight_layout(); plt.show()

## 2. Time-based feature engineering

In [ ]:
df['day_of_week'] = df['Date'].dt.dayofweek
df['day_of_month'] = df['Date'].dt.day
df['month'] = df['Date'].dt.month
df['quarter'] = df['Date'].dt.quarter
df['year'] = df['Date'].dt.year
df['trend'] = np.arange(len(df))
df['lag_1'] = df['Sales'].shift(1)
df['lag_7'] = df['Sales'].shift(7)
df['lag_14'] = df['Sales'].shift(14)
df['rolling_7'] = df['Sales'].shift(1).rolling(7).mean()
df['rolling_30'] = df['Sales'].shift(1).rolling(30).mean()
df = df.dropna().reset_index(drop=True)
df.head()

## 3. Train/test split and model

In [ ]:
features = ['day_of_week','day_of_month','month','quarter','year','trend',
            'lag_1','lag_7','lag_14','rolling_7','rolling_30']
split = int(len(df) * 0.80)
train, test = df.iloc[:split], df.iloc[split:]

model = RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=2, n_jobs=-1)
model.fit(train[features], train['Sales'])
pred = model.predict(test[features])

mae = mean_absolute_error(test['Sales'], pred)
rmse = np.sqrt(mean_squared_error(test['Sales'], pred))
mape = np.mean(np.abs((test['Sales'] - pred) / test['Sales'])) * 100
print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'MAPE: {mape:.2f}%')

## 4. Business-friendly forecast visualization

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(train['Date'].tail(120), train['Sales'].tail(120), label='Historical Sales')
plt.plot(test['Date'], test['Sales'], label='Actual Test Sales')
plt.plot(test['Date'], pred, label='Forecast')
plt.title('Sales Forecast - Historical vs Actual vs Predicted')
plt.xlabel('Date'); plt.ylabel('Sales'); plt.legend()
plt.tight_layout(); plt.show()

## 5. Feature importance and business interpretation

In [ ]:
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
display(importance.head(8))

plt.figure(figsize=(9,5))
importance.head(8).sort_values().plot(kind='barh')
plt.title('Top Forecast Features')
plt.xlabel('Importance'); plt.tight_layout(); plt.show()

print('Average daily sales:', round(df['Sales'].mean(), 2))
print('Highest average-sales month:', int(df.groupby('month')['Sales'].mean().idxmax()))
print('Business use: support inventory, staffing and promotion planning.')